# 01 · Data Ingestion — HR Analytics (2M rows)

> **Stage 1 (STRUCTURE.md).** Acquire the raw data, validate its schema, log metadata, and
> reconcile the raw vs. cleaned files. Raw stays immutable in `archive/`; we emit a
> memory-optimized Parquet for downstream stages.

**What this notebook establishes**
- The cleaned file is exactly **2,000,000 × 16** with **zero** missing values.
- The raw file is **2,000,000 × 15** (no `Hire_Year`) and carries the documented defects.
- Both files describe the **same employees** (identical `Employee_ID` set) → a valid raw→clean pair.

In [1]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import mck_style
mck_style.apply()
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

ARCH = "../archive"; PROC = "../data/processed"; FIG = "../reports/figures"
os.makedirs(PROC, exist_ok=True); os.makedirs(FIG, exist_ok=True)
np.random.seed(42)
print("Setup complete · pandas", pd.__version__)

Setup complete · pandas 3.0.3


### 1.1 Load the cleaned file with explicit dtypes

Explicit dtypes keep 2M rows in a few hundred MB and catch silent coercion early.

In [2]:
dtypes = {
    "Employee_ID": "string", "Full_Name": "string",
    "Department": "category", "Job_Title": "category",
    "Performance_Rating": "category", "Status": "category",
    "Work_Mode": "category", "Country": "category", "City": "category",
    "Job_Level": "category",
    "Experience_Years": "int16", "Salary": "float32",
    "Year": "int16", "Age": "int8", "Hire_Year": "int16",
}
clean = pd.read_csv(f"{ARCH}/hr_clean.csv", dtype=dtypes, parse_dates=["Hire_Date"])
print("Cleaned shape:", clean.shape)
print("Memory (MB):", round(clean.memory_usage(deep=True).sum()/1e6, 1))
clean.head(3)

Cleaned shape: (2000000, 16)
Memory (MB): 140.9


,Employee_ID,Full_Name,Department,Job_Title,Hire_Date,Performance_Rating,Experience_Years,Status,Work_Mode,Salary,Year,Country,City,Age,Job_Level,Hire_Year
0,EMP0000001,Heinz-Georg Eimer,Sales,Business Development,2023-01-31,Satisfactory,8,Active,On-site,"92,992.00",2023,Germany,Munich,32,Mid,2023
1,EMP0000002,Maartje van den Nuwenhuysen-Geertsen,HR,HR Manager,2008-11-07,Good,11,Active,On-site,"112,318.00",2008,Netherlands,Eindhoven,43,Senior,2008
2,EMP0000003,Sara Sureda Figueroa,HR,Talent Specialist,2016-03-19,Needs Improvement,15,Active,On-site,"111,121.00",2016,Spain,Seville,43,Senior,2016


### 1.2 Schema validation against the data dictionary (assessment §1)

In [3]:
EXPECTED = ["Employee_ID","Full_Name","Department","Job_Title","Hire_Date",
            "Performance_Rating","Experience_Years","Status","Work_Mode","Salary",
            "Year","Country","City","Age","Job_Level","Hire_Year"]
assert list(clean.columns) == EXPECTED, "Schema drift!"
assert clean.shape == (2_000_000, 16), "Row/col count mismatch!"
nulls = int(clean.isnull().sum().sum())
assert nulls == 0, f"Expected 0 nulls, found {nulls}"
print("Schema OK · 16 columns · 2,000,000 rows · 0 nulls")
clean.dtypes

Schema OK · 16 columns · 2,000,000 rows · 0 nulls


Employee_ID                   string
Full_Name                     string
Department                  category
Job_Title                   category
Hire_Date             datetime64[us]
Performance_Rating          category
Experience_Years               int16
Status                      category
Work_Mode                   category
Salary                       float32
Year                           int16
Country                     category
City                        category
Age                             int8
Job_Level                   category
Hire_Year                      int16
dtype: object

### 1.3 Metadata log (source · timestamp · counts · hash)

Immutability + lineage per STRUCTURE.md Stage 1.

In [4]:
import hashlib, datetime
def file_md5(path, chunk=8*1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

meta = {
    "source_file": "archive/hr_clean.csv",
    "pulled_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "rows": len(clean), "cols": clean.shape[1],
    "size_MB": round(os.path.getsize(f"{ARCH}/hr_clean.csv")/1e6, 1),
    "md5": file_md5(f"{ARCH}/hr_clean.csv"),
}
pd.Series(meta).to_frame("value")

,value
source_file,archive/hr_clean.csv
pulled_at,2026-07-19T20:09:33
rows,2000000
cols,16
size_MB,270.20
md5,51cb1a21ee403bd8e417e20d34dd3465


### 1.4 Raw file — schema & documented defects

We read only the columns we need to profile defects, to keep memory low.

In [5]:
raw_cols = pd.read_csv(f"{ARCH}/hr_raw.csv", nrows=0).columns.tolist()
print("Raw columns (15):", raw_cols)
print("Missing from raw vs clean:", set(EXPECTED) - set(raw_cols))

raw_probe = pd.read_csv(f"{ARCH}/hr_raw.csv",
    usecols=["Employee_ID","Performance_Rating","Salary","Age","Experience_Years"])
defects = {
    "Perf_Rating nulls":        int(raw_probe["Performance_Rating"].isnull().sum()),
    "Salary <= 0":              int((raw_probe["Salary"] <= 0).sum()),
    "Salary minimum":           float(raw_probe["Salary"].min()),
    "Age-Exp gap < 22 (rows)":  int(((raw_probe["Age"] - raw_probe["Experience_Years"]) < 22).sum()),
    "Age-Exp gap minimum":      int((raw_probe["Age"] - raw_probe["Experience_Years"]).min()),
}
pd.Series(defects).to_frame("raw defect")

Raw columns (15): ['Employee_ID', 'Full_Name', 'Department', 'Job_Title', 'Hire_Date', 'Performance_Rating', 'Experience_Years', 'Status', 'Work_Mode', 'Salary', 'Year', 'Country', 'City', 'Age', 'Job_Level']
Missing from raw vs clean: {'Hire_Year'}


,raw defect
Perf_Rating nulls,"3,333.00"
Salary <= 0,"3,333.00"
Salary minimum,"-99,932.00"
Age-Exp gap < 22 (rows),"164,794.00"
Age-Exp gap minimum,20.00


### 1.5 Raw ↔ clean reconciliation

Same grain, same employees → a legitimate before/after pair.

In [6]:
same_ids = set(raw_probe["Employee_ID"]) == set(clean["Employee_ID"])
print("Employee_ID sets identical:", same_ids)
print("Raw rows:", len(raw_probe), "| Clean rows:", len(clean))
assert same_ids and len(raw_probe) == len(clean)
del raw_probe

Employee_ID sets identical: True
Raw rows: 2000000 | Clean rows: 2000000


### 1.6 Persist a memory-optimized Parquet for downstream stages

In [7]:
clean.to_parquet(f"{PROC}/hr_clean.parquet", index=False)
print("Wrote", f"{PROC}/hr_clean.parquet",
      "·", round(os.path.getsize(f'{PROC}/hr_clean.parquet')/1e6,1), "MB")

Wrote ../data/processed/hr_clean.parquet · 60.6 MB


### Stage 1 — Gate checklist ✅
- [x] Raw kept immutable in `archive/`; optimized Parquet written to `data/processed/`
- [x] Source, timestamp, row/col counts, and MD5 logged
- [x] Schema matches the 16-column dictionary; **0 nulls** confirmed
- [x] Raw (15 cols, defects present) reconciles to clean on `Employee_ID`

→ Proceed to **02 · Cleaning**.